In [ ]:
#Generic settings for the plots

import scienceplots
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams['axes.grid'] = False


mpl.rcParams['xtick.direction'] = 'in'
mpl.rcParams['ytick.direction'] = 'in'


mpl.rcParams['xtick.top'] = True
mpl.rcParams['xtick.bottom'] = True
mpl.rcParams['ytick.left'] = True
mpl.rcParams['ytick.right'] = True


mpl.rcParams['xtick.minor.visible'] = True
mpl.rcParams['ytick.minor.visible'] = True


mpl.rcParams['xtick.major.top'] = True
mpl.rcParams['xtick.major.bottom'] = True
mpl.rcParams['ytick.major.left'] = True
mpl.rcParams['ytick.major.right'] = True

mpl.rcParams['xtick.minor.top'] = True
mpl.rcParams['xtick.minor.bottom'] = True
mpl.rcParams['ytick.minor.left'] = True
mpl.rcParams['ytick.minor.right'] = True

In [ ]:
# Import Directories
import json
from pathlib import Path

# Find project root
try:
    start_dir = Path(__file__).resolve().parent
except NameError:
    start_dir = Path.cwd()

project_root = next(
    (p for p in [start_dir, *start_dir.parents]
     if all((p / folder).is_dir() for folder in ["FeMP", "ACMP", "NiMP"])),
    None
)

if project_root is None:
    raise FileNotFoundError("Project root not found.")

# Directories
dir_Fe = project_root / "FeMP" / "Results_TEA"
dir_AC = project_root / "ACMP" / "Results_TEA"
dir_Ni = project_root / "NiMP" / "Results_TEA"

# JSON files
file_direct_costs_Fe   = dir_Fe / "Direct_costs_avg.json"
file_direct_costs_AC   = dir_AC / "Direct_costs_avg.json"
file_direct_costs_Ni   = dir_Ni / "Direct_costs_avg.json"

file_fixed_capital_Fe  = dir_Fe / "Fixed_capital_avg.json"
file_fixed_capital_AC  = dir_AC / "Fixed_capital_avg.json"
file_fixed_capital_Ni  = dir_Ni / "Fixed_capital_avg.json"

file_variable_opex_Fe  = dir_Fe / "Variable_opex_avg.json"
file_variable_opex_AC  = dir_AC / "Variable_opex_avg.json"
file_variable_opex_Ni  = dir_Ni / "Variable_opex_avg.json"

file_fixed_opex_Fe     = dir_Fe / "Fixed_opex_avg.json"
file_fixed_opex_AC     = dir_AC / "Fixed_opex_avg.json"
file_fixed_opex_Ni     = dir_Ni / "Fixed_opex_avg.json"

file_levelized_cost_Fe = dir_Fe / "LCOH_avg.json"
file_levelized_cost_AC = dir_AC / "LCOH_avg.json"
file_levelized_cost_Ni = dir_Ni / "LCOH_avg.json"

# Load JSON
with open(file_direct_costs_Fe, "r") as f:   direct_costs_Fe = json.load(f)
with open(file_direct_costs_AC, "r") as f:   direct_costs_AC = json.load(f)
with open(file_direct_costs_Ni, "r") as f:   direct_costs_Ni = json.load(f)

with open(file_fixed_capital_Fe, "r") as f:  fixed_capital_Fe = json.load(f)
with open(file_fixed_capital_AC, "r") as f:  fixed_capital_AC = json.load(f)
with open(file_fixed_capital_Ni, "r") as f:  fixed_capital_Ni = json.load(f)

with open(file_variable_opex_Fe, "r") as f:  variable_opex_Fe = json.load(f)
with open(file_variable_opex_AC, "r") as f:  variable_opex_AC = json.load(f)
with open(file_variable_opex_Ni, "r") as f:  variable_opex_Ni = json.load(f)

with open(file_fixed_opex_Fe, "r") as f:     fixed_opex_Fe = json.load(f)
with open(file_fixed_opex_AC, "r") as f:     fixed_opex_AC = json.load(f)
with open(file_fixed_opex_Ni, "r") as f:     fixed_opex_Ni = json.load(f)

with open(file_levelized_cost_Fe, "r") as f: levelized_cost_Fe = json.load(f)
with open(file_levelized_cost_AC, "r") as f: levelized_cost_AC = json.load(f)
with open(file_levelized_cost_Ni, "r") as f: levelized_cost_Ni = json.load(f)

In [ ]:
#COSTS BREAKDOWN

import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import scienceplots

from pathlib import Path
from matplotlib.ticker import MaxNLocator, AutoMinorLocator
from matplotlib.patches import Patch


plt.style.use(["science", "ieee"])

base_font_old = 7.5
base_font_new = 10.5
legend_font_new = 8.7
scale_factor = base_font_new / base_font_old

mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times"],
    "font.size": base_font_new,
    "axes.labelsize": base_font_new,
    "xtick.labelsize": base_font_new,
    "ytick.labelsize": base_font_new,
    "legend.fontsize": legend_font_new,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "axes.grid": False,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

labels = [
    "Fe\n725 $^\\circ$C",
    "AC\n825 $^\\circ$C",
    "Ni\n600 $^\\circ$C"
]


def scale_breakdown(data, divisor=1.0):
    return {
        name: float(value) / divisor
        for name, value in data.items()
    }


def group_direct_costs(data):

    grouped = {}

    for name, value in data.items():

        name_upper = name.upper().strip()

        if re.match(r"^COMP[-_ ]?\d+$", name_upper):
            group = "Compressors"

        elif re.match(r"^MTR[-_ ]?\d+$", name_upper):
            group = "Motors"

        elif re.match(r"^HX[-_ ]?\d+$", name_upper):
            group = "Heat exchangers"

        elif re.match(r"^COOLER[-_ ]?\d+$", name_upper):
            group = "Coolers"

        elif "REACTOR" in name_upper or name_upper in {"CSTR", "PFR"}:
            group = "Reactor"

        elif "FURNACE" in name_upper:
            group = "Furnace"

        elif "CYCLONE" in name_upper:
            group = "Cyclone"

        elif "TURBINE" in name_upper:
            group = "Turbine"

        elif "PUMP" in name_upper:
            group = "Pump"

        elif "CO2" in name_upper and "CAPTURE" in name_upper:
            group = "CO2-capture"

        elif "PSA" in name_upper:
            group = "PSA"

        elif "GENERATOR" in name_upper:
            group = "Generator"

        else:
            group = name

        grouped[group] = grouped.get(group, 0.0) + float(value)

    return grouped

# Legend adjustment
def normalize_fixed_capital_components(data):

    normalized = {}

    for name, value in data.items():

        name_upper = name.upper().strip()
        name_compact = re.sub(r"[^A-Z0-9]", "", name_upper)

        if name_compact == "ISBL":
            new_name = "ISBL"

        elif name_compact == "OSBL":
            new_name = "OSBL"

        elif (
            name_compact == "DE"
            or (
                "DESIGN" in name_upper
                and "ENGINEER" in name_upper
            )
        ):
            new_name = "Design and Engineering"

        elif "CONTINGENCY" in name_upper:
            new_name = "Contingency"

        elif (
            "ADDITIONAL" in name_upper
            and "CAPEX" in name_upper
        ):
            new_name = "Additional CAPEX"

        else:
            new_name = name

        normalized[new_name] = (
            normalized.get(new_name, 0.0)
            + float(value)
        )

    return normalized

#Data collection

direct_data = {
    labels[0]: scale_breakdown(group_direct_costs(direct_costs_Fe), 1e6),
    labels[1]: scale_breakdown(group_direct_costs(direct_costs_AC), 1e6),
    labels[2]: scale_breakdown(group_direct_costs(direct_costs_Ni), 1e6),
}

fixed_capital_data_plot = {
    labels[0]: scale_breakdown(
        normalize_fixed_capital_components(fixed_capital_Fe), 1e6
    ),
    labels[1]: scale_breakdown(
        normalize_fixed_capital_components(fixed_capital_AC), 1e6
    ),
    labels[2]: scale_breakdown(
        normalize_fixed_capital_components(fixed_capital_Ni), 1e6
    ),
}

variable_opex_data_plot = {
    labels[0]: scale_breakdown(variable_opex_Fe, 1e6),
    labels[1]: scale_breakdown(variable_opex_AC, 1e6),
    labels[2]: scale_breakdown(variable_opex_Ni, 1e6),
}

fixed_opex_data_plot = {
    labels[0]: scale_breakdown(fixed_opex_Fe, 1e6),
    labels[1]: scale_breakdown(fixed_opex_AC, 1e6),
    labels[2]: scale_breakdown(fixed_opex_Ni, 1e6),
}


def get_component_order(data):

    totals = {}

    for breakdown in data.values():

        for component, value in breakdown.items():

            totals[component] = (
                totals.get(component, 0.0)
                + abs(float(value))
            )

    return sorted(
        totals.keys(),
        key=lambda component: totals[component],
        reverse=True
    )

#Legend

def get_fe_legend_order(data):

    fe_breakdown = data[labels[0]]

    fe_order = sorted(
        fe_breakdown.keys(),
        key=lambda component: abs(float(fe_breakdown[component])),
        reverse=True
    )

    all_components = []

    for breakdown in data.values():

        for component in breakdown:

            if component not in all_components:
                all_components.append(component)

    missing_components = [
        component
        for component in all_components
        if component not in fe_order
    ]

    missing_components = sorted(
        missing_components,
        key=lambda component: sum(
            abs(float(breakdown.get(component, 0.0)))
            for breakdown in data.values()
        ),
        reverse=True
    )

    return fe_order + missing_components


# DIRECT EQUIPMENT COSTS

direct_palette = [
    "#D2A52A",
    "#3D7C4A",
    "#138080",
    "#3D68A8",
    "#8A903D",
    "#72589A",
    "#5C874C",
    "#237A66",
    "#327687",
    "#4D6F95",
    "#657A66",
    "#6D697F",
]

direct_color_order = get_component_order(
    direct_data
)

direct_colors = {
    component:
        direct_palette[
            i % len(direct_palette)
        ]
    for i, component
    in enumerate(direct_color_order)
}


# FIXED CAPITAL
fixed_capital_palette = [
    "#285A8E",
    "#3A8F86",
    "#91A33B",
    "#6E5A9E",
    "#9ABBD0",
]

fixed_capital_color_order = get_component_order(
    fixed_capital_data_plot
)

fixed_capital_colors = {
    component:
        fixed_capital_palette[
            i % len(fixed_capital_palette)
        ]
    for i, component
    in enumerate(fixed_capital_color_order)
}


# VARIABLE OPEX
variable_opex_palette = [
    "#D9A520",
    "#C6533D",
    "#76528F",
    "#DE8431",
    "#963F59",
    "#B56F63",
]

variable_opex_color_order = get_component_order(
    variable_opex_data_plot
)

variable_opex_colors = {
    component:
        variable_opex_palette[
            i % len(variable_opex_palette)
        ]
    for i, component
    in enumerate(variable_opex_color_order)
}


# FIXED OPEX
fixed_opex_palette = [
    "#D9C2EA",
    "#55308E",
    "#2868A4",
    "#A53E7E",
    "#21877E",
    "#7E57B5",
    "#34476D",
    "#C073A4",
    "#4A88B6",
    "#713C91",
    "#34798F",
    "#94546F",
    "#576CA0",
    "#88728E",
]

fixed_opex_color_order = get_component_order(
    fixed_opex_data_plot
)

fixed_opex_colors = {
    component:
        fixed_opex_palette[
            i % len(fixed_opex_palette)
        ]
    for i, component
    in enumerate(fixed_opex_color_order)
}


def complete_color_map(
    data,
    preferred_colors=None
):

    preferred_colors = preferred_colors or {}

    components = []

    for breakdown in data.values():

        for component in breakdown:

            if component not in components:
                components.append(component)

    color_map = dict(
        preferred_colors
    )

    palette = list(
        plt.cm.tab20.colors
    )

    index = 0

    for component in components:

        if component not in color_map:

            color_map[component] = (
                palette[
                    index % len(palette)
                ]
            )

            index += 1

    return color_map


def clean_label(name):

    replacements = {
        "electricity":
            "Electricity",

        "methane_feed":
            "Methane feed",

        "catalyst_feed":
            "Catalyst feed",

        "cooling_water":
            "Cooling water",

        "CO2_tax":
            r"CO$_2$ tax",

        "CO2 tax":
            r"CO$_2$ tax",

        "carbon_disposal_fee":
            "Carbon disposal fee",

        "CO2-capture":
            r"CO$_2$ capture",

        "D&E":
            "Design and Engineering",

        r"D\&E":
            "Design and Engineering",

        "Design & Engineering":
            "Design and Engineering",

        r"Design \& Engineering":
            "Design and Engineering",

        "R&D":
            r"R\&D",

        r"R\&D":
            r"R\&D",
    }

    label = replacements.get(
        name,
        name
    )

    if label == r"R\&D":
        return label

    label = label.replace(
        r"\&",
        " and "
    )

    label = label.replace(
        "&",
        " and "
    )

    label = re.sub(
        r"\s+",
        " ",
        label
    ).strip()

    return label

# Stacked Bars

def plot_sorted_stacked_bars(
    data,
    ylabel,
    preferred_colors=None,
    y_max=None,
    legend_fontsize=legend_font_new
):

    color_map = complete_color_map(
        data,
        preferred_colors
    )


    legend_order = get_fe_legend_order(data)

    fig, ax = plt.subplots(
        figsize=(
            3.6 * scale_factor,
            2.8 * scale_factor
        )
    )

    x_positions = np.arange(len(data))
    bar_width = 0.62

    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    max_positive = 0.0
    min_negative = 0.0


    for x_position, (_, components) in enumerate(
        data.items()
    ):

        positive = sorted(
            [
                (component, float(value))
                for component, value
                in components.items()
                if float(value) >= 0
            ],
            key=lambda item: item[1],
            reverse=True
        )

        negative = sorted(
            [
                (component, float(value))
                for component, value
                in components.items()
                if float(value) < 0
            ],
            key=lambda item: abs(item[1]),
            reverse=True
        )

        positive_bottom = 0.0

        for component, value in positive:

            ax.bar(
                x_position,
                value,
                width=bar_width,
                bottom=positive_bottom,
                color=color_map[component],
                edgecolor="black",
                linewidth=0.8,
                zorder=2
            )

            positive_bottom += value


        negative_bottom = 0.0

        for component, value in negative:

            ax.bar(
                x_position,
                value,
                width=bar_width,
                bottom=negative_bottom,
                color=color_map[component],
                edgecolor="black",
                linewidth=0.8,
                zorder=3
            )

            negative_bottom += value

        max_positive = max(
            max_positive,
            positive_bottom
        )

        min_negative = min(
            min_negative,
            negative_bottom
        )


    ax.set_ylabel(
        ylabel,
        fontsize=base_font_new,
        labelpad=7
    )

    ax.set_xticks(
        x_positions
    )

    ax.set_xticklabels(
        data.keys(),
        fontsize=base_font_new
    )

    ax.set_xlim(
        -0.65,
        len(data) - 0.35
    )


    if y_max is None:
        y_max = (
            max_positive
            * 1.08
        )

    if min_negative < 0:

        ax.set_ylim(
            min_negative * 1.15,
            y_max
        )

        ax.axhline(
            0,
            color="black",
            linewidth=1.0,
            zorder=5
        )

    else:

        ax.set_ylim(
            0,
            y_max
        )


    ax.yaxis.set_major_locator(
        MaxNLocator(
            nbins=7
        )
    )

    ax.yaxis.set_minor_locator(
        AutoMinorLocator(5)
    )


    ax.tick_params(
        axis="both",
        which="major",
        direction="in",
        top=True,
        right=True,
        length=4.9,
        width=1.1,
        labelsize=base_font_new,
        pad=7
    )

    ax.tick_params(
        axis="both",
        which="minor",
        direction="in",
        top=True,
        right=True,
        length=2.8,
        width=0.8
    )


    for spine in ax.spines.values():

        spine.set_color(
            "black"
        )

        spine.set_linewidth(
            1.1
        )


    legend_handles = [
        Patch(
            facecolor=color_map[
                component
            ],
            edgecolor="black",
            linewidth=0.8,
            label=clean_label(
                component
            )
        )
        for component
        in legend_order
    ]

    ax.legend(
        handles=legend_handles,
        frameon=False,
        fontsize=legend_fontsize,
        ncol=1,
        loc="center left",
        bbox_to_anchor=(
            1.02,
            0.5
        )
    )

    plt.tight_layout()

    return fig, ax


# 1. DIRECT EQUIPMENT COSTS
fig_direct, ax_direct = plot_sorted_stacked_bars(
    data=direct_data,
    ylabel=r"Direct equipment costs / [millions US\$]",
    preferred_colors=direct_colors,
    legend_fontsize=legend_font_new
)


# 2. FIXED CAPITAL
fig_capital, ax_capital = plot_sorted_stacked_bars(
    data=fixed_capital_data_plot,
    ylabel=r"Fixed capital costs / [millions US\$]",
    preferred_colors=fixed_capital_colors,
    legend_fontsize=legend_font_new
)


# 3. VARIABLE OPEX
fig_variable, ax_variable = plot_sorted_stacked_bars(
    data=variable_opex_data_plot,
    ylabel=(
        r"Annual variable OPEX / "
        r"[millions US\$\,$\cdot$\,year$^{\mbox{-}1}$]"
    ),
    preferred_colors=variable_opex_colors,
    legend_fontsize=legend_font_new
)


# 4. FIXED OPEX
fig_fixed, ax_fixed = plot_sorted_stacked_bars(
    data=fixed_opex_data_plot,
    ylabel=(
        r"Annual fixed OPEX / "
        r"[millions US\$\,$\cdot$\,year$^{\mbox{-}1}$]"
    ),
    preferred_colors=fixed_opex_colors,
    legend_fontsize=legend_font_new
)

# Save as pdf

fig_direct.savefig("direct_equipment_costs.pdf",
    format="pdf"
)

fig_capital.savefig("fixed_capital_costs.pdf",
    format="pdf"
)

fig_variable.savefig("annual_variable_opex.pdf",
    format="pdf"
)

fig_fixed.savefig("annual_fixed_opex.pdf",
    format="pdf"
)

plt.show()

In [ ]:
#LCOH BASELINE
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import scienceplots

from matplotlib.ticker import MultipleLocator
from matplotlib.patches import Patch

plt.style.use(["science", "ieee"])

base_font_old = 7.5
base_font_new = 10.5
legend_font_new = 8.7
scale_factor = base_font_new / base_font_old

mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times"],
    "font.size": base_font_new,
    "axes.labelsize": base_font_new,
    "xtick.labelsize": base_font_new,
    "ytick.labelsize": base_font_new,
    "legend.fontsize": legend_font_new,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "axes.grid": False,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

labels = [
    "Fe\n725 $^\\circ$C",
    "AC\n825 $^\\circ$C",
    "Ni\n600 $^\\circ$C"
]

def scale_breakdown(data, divisor=1.0):
    return {name: float(value) / divisor for name, value in data.items()}

levelized_cost_data_plot = {
    labels[0]: scale_breakdown(levelized_cost_Fe),
    labels[1]: scale_breakdown(levelized_cost_AC),
    labels[2]: scale_breakdown(levelized_cost_Ni),
}


levelized_colors = {
    "CAPEX":         "#F4B5A7",
    "Variable OPEX": "#B5E3C6",
    "Fixed OPEX":    "#D8D1DF",
    "Carbon sales":  "#BDBDBD",
}

def complete_color_map(data, preferred_colors=None):

    preferred_colors = preferred_colors or {}

    components = []

    for breakdown in data.values():
        for component in breakdown:
            if component not in components:
                components.append(component)

    color_map = dict(preferred_colors)
    palette = list(plt.cm.tab20.colors)

    index = 0

    for component in components:

        if component not in color_map:
            color_map[component] = palette[index % len(palette)]
            index += 1

    return color_map

def clean_label(name):

    replacements = {
        "electricity":            "Electricity",
        "methane_feed":           "Methane feed",
        "catalyst_feed":          "Catalyst feed",
        "cooling_water":          "Cooling water",
        "CO2_tax":                r"CO$_2$ tax",
        "CO2 tax":                r"CO$_2$ tax",
        "carbon_disposal_fee":    "Carbon disposal fee",
        "CO2-capture":            r"CO$_2$ capture",
        "D&E":                    "Design and Engineering",
        r"D\&E":                  "Design and Engineering",
        "Design & Engineering":   "Design and Engineering",
        r"Design \& Engineering": "Design and Engineering",
        "R&D":                    r"R\&D",
        r"R\&D":                  r"R\&D",
    }

    label = replacements.get(name, name)

    if label == r"R\&D":
        return label

    label = label.replace(r"\&", " and ")
    label = label.replace("&", " and ")
    label = re.sub(r"\s+", " ", label).strip()

    return label

#Stacked bar
def plot_sorted_stacked_bars(
    data,
    ylabel,
    preferred_colors=None,
    y_max=None,
    y_major_step=None,
    y_minor_step=None,
    shift_by_negative=False,
    legend_inside=False,
    legend_fontsize=legend_font_new
):

    color_map = complete_color_map(data, preferred_colors)

    fig, ax = plt.subplots(
        figsize=(3.6 * scale_factor, 2.8 * scale_factor)
    )

    x_positions = np.arange(len(data))
    bar_width = 0.62

    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    max_positive = 0.0
    min_negative = 0.0

    #Function to calculate positive and negative (side revenue) values
    for x_position, (_, components) in enumerate(data.items()):

        positive = sorted(
            [(name, value) for name, value in components.items() if value >= 0],
            key=lambda item: item[1],
            reverse=True
        )

        negative = sorted(
            [(name, value) for name, value in components.items() if value < 0],
            key=lambda item: abs(item[1]),
            reverse=True
        )

        negative_total = sum(value for _, value in negative)

        positive_bottom = negative_total if shift_by_negative else 0.0

        for component, value in positive:

            ax.bar(
                x_position,
                value,
                width=bar_width,
                bottom=positive_bottom,
                color=color_map[component],
                edgecolor="black",
                linewidth=0.8,
                zorder=2
            )

            positive_bottom += value

        negative_bottom = 0.0

        for component, value in negative:

            ax.bar(
                x_position,
                value,
                width=bar_width,
                bottom=negative_bottom,
                color=color_map[component],
                edgecolor="black",
                linewidth=0.8,
                zorder=3
            )

            negative_bottom += value


        max_positive = max(
            max_positive,
            positive_bottom
        )

        min_negative = min(
            min_negative,
            negative_bottom
        )

    ax.set_ylabel(
        ylabel,
        fontsize=base_font_new,
        labelpad=7
    )

    ax.set_xticks(x_positions)

    ax.set_xticklabels(
        data.keys(),
        fontsize=base_font_new
    )

    ax.set_xlim(
        -0.65,
        len(data) - 0.35
    )

    if y_max is None:
        y_max = max_positive * 1.08

    if min_negative < 0:

        ax.set_ylim(
            min_negative * 1.15,
            y_max
        )

        ax.axhline(
            0,
            color="black",
            linewidth=1.0,
            zorder=5
        )

    else:

        ax.set_ylim(
            0,
            y_max
        )

    if y_major_step is not None:
        ax.yaxis.set_major_locator(
            MultipleLocator(y_major_step)
        )

    if y_minor_step is not None:
        ax.yaxis.set_minor_locator(
            MultipleLocator(y_minor_step)
        )

    ax.tick_params(
        axis="both",
        which="major",
        direction="in",
        top=True,
        right=True,
        length=4.9,
        width=1.1,
        labelsize=base_font_new,
        pad=7
    )

    ax.tick_params(
        axis="both",
        which="minor",
        direction="in",
        top=True,
        right=True,
        length=2.8,
        width=0.8
    )

    for spine in ax.spines.values():
        spine.set_color("black")
        spine.set_linewidth(1.1)

    components_present = []

    for breakdown in data.values():
        for component in breakdown:
            if component not in components_present:
                components_present.append(component)

    legend_handles = [
        Patch(
            facecolor=color_map[component],
            edgecolor="black",
            linewidth=0.8,
            label=clean_label(component)
        )
        for component in components_present
    ]

    if legend_inside:

        ax.legend(
            handles=legend_handles,
            frameon=False,
            fontsize=legend_fontsize,
            ncol=1,
            loc="upper right"
        )

    else:

        ax.legend(
            handles=legend_handles,
            frameon=False,
            fontsize=legend_fontsize,
            ncol=1,
            loc="center left",
            bbox_to_anchor=(1.02, 0.5)
        )

    plt.tight_layout()

    return fig, ax

#Final figure
fig_levelized, ax_levelized = plot_sorted_stacked_bars(
    data=levelized_cost_data_plot,
    ylabel=r"LCOH / [US\$\,$\cdot$\,kg$^{\mbox{-}1}$]",
    preferred_colors=levelized_colors,
    y_major_step=5,
    y_minor_step=1,
    shift_by_negative=True,
    legend_inside=True,
    legend_fontsize=10.0
)

fig_levelized.savefig("levelized_cost_breakdown.pdf",
    format="pdf"
)

plt.show()
